<a href="https://colab.research.google.com/github/ali-liyakat/PyTorch_for_Deep_Learning/blob/main/04_pytorch_nn_module.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## PyTorch nn Module

In [1]:
import torch
import torch.nn as nn

### Create Model Class

In [2]:
class Model(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(num_features, 3),
            nn.ReLU(),
            nn.Linear(3, 1),
            nn.Sigmoid()
        )

    def forward(self, features):
      out = self.network(features)
      return out

### Create Dataset

In [3]:
features = torch.rand(10, 5)

# create Model
model = Model(features.shape[1])

# forward pass
out = model(features)
out

tensor([[0.5323],
        [0.5217],
        [0.5502],
        [0.5241],
        [0.5168],
        [0.5374],
        [0.5326],
        [0.5258],
        [0.5178],
        [0.5140]], grad_fn=<SigmoidBackward0>)

In [4]:
from torchinfo import summary
summary(model, input_size=(10, 5))

Layer (type:depth-idx)                   Output Shape              Param #
Model                                    [10, 1]                   --
├─Sequential: 1-1                        [10, 1]                   --
│    └─Linear: 2-1                       [10, 3]                   18
│    └─ReLU: 2-2                         [10, 3]                   --
│    └─Linear: 2-3                       [10, 1]                   4
│    └─Sigmoid: 2-4                      [10, 1]                   --
Total params: 22
Trainable params: 22
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

### Breast cancer dataset

In [5]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [6]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [7]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2, random_state=42)

In [9]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [10]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [11]:
X_train_tensor = torch.from_numpy(X_train).float()
X_test_tensor = torch.from_numpy(X_test).float()
y_train_tensor = torch.from_numpy(y_train).float().view(-1, 1)
y_test_tensor = torch.from_numpy(y_test).float().view(-1, 1)

In [12]:
class BreastCancer(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(num_features, 1),
            nn.Sigmoid()
        )

    def forward(self, features):
      out = self.network(features)
      return out

In [13]:
learning_rate = 0.01
epochs = 25
loss_function = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [14]:
# create model
model = BreastCancer(X_train_tensor.shape[1])

# define loop
for epoch in range(epochs):

    # forward Pass
    y_pred = model(X_train_tensor)

    # loss calculate
    loss = loss_function(y_pred, y_train_tensor)

    # Clear gradients
    optimizer.zero_grad()


    # backward Pass
    loss.backward()

    # parameter update
    optimizer.step()

    print(f'Epoch: {epoch+1}, Loss: {loss.item()}')

Epoch: 1, Loss: 0.9204713106155396
Epoch: 2, Loss: 0.9204713106155396
Epoch: 3, Loss: 0.9204713106155396
Epoch: 4, Loss: 0.9204713106155396
Epoch: 5, Loss: 0.9204713106155396
Epoch: 6, Loss: 0.9204713106155396
Epoch: 7, Loss: 0.9204713106155396
Epoch: 8, Loss: 0.9204713106155396
Epoch: 9, Loss: 0.9204713106155396
Epoch: 10, Loss: 0.9204713106155396
Epoch: 11, Loss: 0.9204713106155396
Epoch: 12, Loss: 0.9204713106155396
Epoch: 13, Loss: 0.9204713106155396
Epoch: 14, Loss: 0.9204713106155396
Epoch: 15, Loss: 0.9204713106155396
Epoch: 16, Loss: 0.9204713106155396
Epoch: 17, Loss: 0.9204713106155396
Epoch: 18, Loss: 0.9204713106155396
Epoch: 19, Loss: 0.9204713106155396
Epoch: 20, Loss: 0.9204713106155396
Epoch: 21, Loss: 0.9204713106155396
Epoch: 22, Loss: 0.9204713106155396
Epoch: 23, Loss: 0.9204713106155396
Epoch: 24, Loss: 0.9204713106155396
Epoch: 25, Loss: 0.9204713106155396


In [15]:
# model evaluation
with torch.no_grad():
    y_pred = model.forward(X_test_tensor)
    y_pred = (y_pred > 0.9).float()

    accuaracy = (y_pred == y_test_tensor).float().mean()
    print(f'Accuracy: {accuaracy.item()}')

Accuracy: 0.6228070259094238
